# Interactive NDCG@10 vs. Sparsity Curves

Plotly side-by-side: Experiment A (shrinking catalogue) vs Experiment B (fixed 62,423-film catalogue) across all models. Saved to `outputs/ndcg_sparsity_curves.html`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

OUTPUT_DIR = '../outputs/'

# load data
df_a = pd.read_csv(OUTPUT_DIR + 'sparsity_expA_all_models.csv')
df_b = pd.read_csv(OUTPUT_DIR + 'sparsity_expB_all_models.csv')

# ensure consistent column names
for df in [df_a, df_b]:
    df['log_n'] = np.log10(df['n_ratings'])

print('Exp A shape:', df_a.shape)
print('Exp B shape:', df_b.shape)
print()
print('Models:', sorted(df_a['model'].unique()))
print('Sparsity levels:', sorted(df_a['sparsity'].unique()))

In [ ]:
# style config
MODEL_ORDER  = ['popularity_baseline', 'svd', 'penalized_svd', 'content_based', 'hybrid']
MODEL_LABELS = {
    'popularity_baseline': 'Popularity Baseline',
    'svd'                : 'SVD',
    'penalized_svd'      : 'Penalised SVD (λ=0.3)',
    'content_based'      : 'Content-Based',
    'hybrid'             : 'Hybrid (α=0.6)',
}
COLOURS = {
    'popularity_baseline': '#636EFA',
    'svd'                : '#EF553B',
    'penalized_svd'      : '#00CC96',
    'content_based'      : '#AB63FA',
    'hybrid'             : '#FFA15A',
}
DASH = {
    'popularity_baseline': 'solid',
    'svd'                : 'dot',
    'penalized_svd'      : 'dash',
    'content_based'      : 'dashdot',
    'hybrid'             : 'longdash',
}

# crossover detection
def find_winner_crossovers(df):
    """
    Return list of (log10_x_cross, model_before, model_after) tuples for each
    point where the best-performing model changes as n_ratings increases.
    Uses linear interpolation in log10(n_ratings) space.
    """
    n_vals = sorted(df['n_ratings'].unique())

    # winner at each n_ratings value
    def winner_at(n):
        sub = df[df['n_ratings'] == n]
        return sub.loc[sub['ndcg'].idxmax(), 'model']

    winners = [(n, winner_at(n)) for n in n_vals]
    crossovers = []

    for i in range(len(winners) - 1):
        n0, w0 = winners[i]
        n1, w1 = winners[i + 1]
        if w0 == w1:
            continue

        x0 = np.log10(n0)
        x1 = np.log10(n1)

        # values for both competing models at both flanking points
        def ndcg(model, n):
            row = df[(df['model'] == model) & (df['n_ratings'] == n)]
            return float(row['ndcg'].values[0]) if len(row) else np.nan

        ya0, ya1 = ndcg(w0, n0), ndcg(w0, n1)
        yb0, yb1 = ndcg(w1, n0), ndcg(w1, n1)

        if any(np.isnan(v) for v in [ya0, ya1, yb0, yb1]):
            crossovers.append(((x0 + x1) / 2, w0, w1))  # midpoint fallback
            continue

        # solve: ya0 + t*(ya1-ya0) = yb0 + t*(yb1-yb0)
        denom = (ya1 - ya0) - (yb1 - yb0)
        if abs(denom) < 1e-10:
            crossovers.append(((x0 + x1) / 2, w0, w1))
        else:
            t = (yb0 - ya0) / denom
            t = float(np.clip(t, 0, 1))
            x_cross = x0 + t * (x1 - x0)
            crossovers.append((x_cross, w0, w1))

    return crossovers


crossovers_a = find_winner_crossovers(df_a)
crossovers_b = find_winner_crossovers(df_b)

print('Exp A crossovers (log10 scale):')
for xc, wa, wb in crossovers_a:
    print(f'  log10(n)={xc:.3f}  (10^x ≈ {10**xc:,.0f})  {wa} → {wb}')
print()
print('Exp B crossovers (log10 scale):')
for xc, wa, wb in crossovers_b:
    print(f'  log10(n)={xc:.3f}  (10^x ≈ {10**xc:,.0f})  {wa} → {wb}')

In [ ]:
# build figure
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Experiment A: Shrinking Catalogue',
        'Experiment B: Fixed Catalogue (62,423 films)',
    ],
    horizontal_spacing=0.10,
)

# track which models have already appeared in the legend (shared across subplots)
legend_shown = set()


def add_model_traces(df, col, crossovers):
    """Add line + CI band for every model, plus crossover lines."""

    for model in MODEL_ORDER:
        sub = df[df['model'] == model].sort_values('log_n')
        if sub.empty:
            continue

        x   = sub['log_n'].tolist()
        y   = sub['ndcg'].tolist()
        sp  = sub['sparsity'].tolist()
        nr  = sub['n_ratings'].tolist()
        ylo = sub['ci_lower'].tolist() if 'ci_lower' in sub.columns else None
        yhi = sub['ci_upper'].tolist() if 'ci_upper' in sub.columns else None

        colour     = COLOURS[model]
        show_leg   = model not in legend_shown
        label      = MODEL_LABELS[model]

        # hover text
        hover = [
            f'<b>{label}</b><br>'
            f'NDCG@10: {yi:.4f}<br>'
            f'95% CI: [{lo:.4f}, {hi:.4f}]<br>'
            f'Sparsity: {si}<br>'
            f'n_ratings: {ni:,}'
            for yi, lo, hi, si, ni in zip(
                y,
                ylo if ylo else [float('nan')] * len(y),
                yhi if yhi else [float('nan')] * len(y),
                sp, nr
            )
        ]

        # CI shaded band (filled area, no hover/legend entry)
        if ylo and yhi:
            fig.add_trace(
                go.Scatter(
                    x=x + x[::-1],
                    y=yhi + ylo[::-1],
                    fill='toself',
                    fillcolor=colour,
                    opacity=0.15,
                    line=dict(width=0),
                    hoverinfo='skip',
                    showlegend=False,
                    name=label + ' CI',
                ),
                row=1, col=col
            )

        # main line
        fig.add_trace(
            go.Scatter(
                x=x, y=y,
                mode='lines+markers',
                name=label,
                line=dict(color=colour, width=2.5, dash=DASH[model]),
                marker=dict(size=8, color=colour,
                            line=dict(width=1, color='white')),
                hovertemplate='%{customdata}<extra></extra>',
                customdata=hover,
                legendgroup=model,
                showlegend=show_leg,
            ),
            row=1, col=col
        )

        if show_leg:
            legend_shown.add(model)

    # crossover vertical lines
    for i, (xc, w_before, w_after) in enumerate(crossovers):
        n_approx = 10 ** xc
        fig.add_vline(
            x=xc,
            line=dict(color='black', dash='dash', width=1.5),
            annotation_text=(
                f'Crossover<br>n≈{n_approx:,.0f}<br>'
                f'{MODEL_LABELS[w_before].split()[0]}→{MODEL_LABELS[w_after].split()[0]}'
            ),
            annotation_position='top right' if i % 2 == 0 else 'top left',
            annotation_font_size=10,
            row=1, col=col
        )


add_model_traces(df_a, col=1, crossovers=crossovers_a)
add_model_traces(df_b, col=2, crossovers=crossovers_b)

# layout
# shared x-axis tick labels (log10 → readable n)
tick_log = [np.log10(v) for v in [1_000, 10_000, 100_000, 1_000_000,
                                   10_000_000, 25_000_000]]
tick_txt = ['1K', '10K', '100K', '1M', '10M', '25M']

axis_common = dict(
    tickvals=tick_log,
    ticktext=tick_txt,
    gridcolor='rgba(200,200,200,0.4)',
    showgrid=True,
    zeroline=False,
    title_font_size=12,
)

fig.update_xaxes(title_text='Number of Ratings (log scale)', **axis_common)
fig.update_yaxes(
    title_text='NDCG@10',
    gridcolor='rgba(200,200,200,0.4)',
    showgrid=True,
    zeroline=True,
    zerolinecolor='rgba(150,150,150,0.5)',
    zerolinewidth=1,
    rangemode='tozero',
    title_font_size=12,
    row=1, col=1
)
# right subplot: no repeated y-axis title
fig.update_yaxes(
    gridcolor='rgba(200,200,200,0.4)',
    showgrid=True,
    zeroline=True,
    zerolinecolor='rgba(150,150,150,0.5)',
    zerolinewidth=1,
    rangemode='tozero',
    row=1, col=2
)

fig.update_layout(
    title=dict(
        text=(
            'NDCG@10 vs. Rating Density — Sparsity Experiments<br>'
            '<sup>Shaded bands = 95% bootstrap CI (n=500 users, 1,000 resamples) · '
            'Dashed lines = crossover between leading models</sup>'
        ),
        x=0.5,
        font_size=15,
    ),
    legend=dict(
        title='Model',
        orientation='h',
        yanchor='bottom',
        y=-0.20,
        xanchor='center',
        x=0.5,
        font_size=12,
        itemsizing='constant',
    ),
    hovermode='closest',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=520,
    width=1200,
    margin=dict(t=100, b=130, l=70, r=40),
)

# save
out_path = OUTPUT_DIR + 'ndcg_sparsity_curves.html'
fig.write_html(out_path, include_plotlyjs='cdn')
print(f'Saved  →  {out_path}')

fig.show()

# Accuracy–Fairness Tradeoff Curve

Catalogue coverage vs. long-tail coverage across λ values for penalized SVD. Saved to `outputs/lambda_tradeoff.html`.

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

OUTPUT_DIR = '../outputs/'

df_t = pd.read_csv(OUTPUT_DIR + 'tradeoff_results.csv')
df_t = df_t.sort_values('lambda').reset_index(drop=True)
print(df_t.to_string(index=False))

# color gradient: blue (λ=0) → red (λ=max)
lam_min  = df_t['lambda'].min()
lam_max  = df_t['lambda'].max()
lam_norm = ((df_t['lambda'] - lam_min) / (lam_max - lam_min)).tolist()

def lerp_color(t, c0=(33, 102, 172), c1=(214, 96, 77)):
    """Interpolate between two RGB colours; t in [0, 1]."""
    r = int(c0[0] + t * (c1[0] - c0[0]))
    g = int(c0[1] + t * (c1[1] - c0[1]))
    b = int(c0[2] + t * (c1[2] - c0[2]))
    return f'rgb({r},{g},{b})'

pt_colors    = [lerp_color(t) for t in lam_norm]
has_pop_rank = 'mean_popularity_rank' in df_t.columns

# hover text
def make_hover(row):
    txt = (
        f"<b>\u03bb = {row['lambda']}</b><br>"
        f"Catalogue Coverage: {row['catalog_coverage']:.2f}%<br>"
        f"Long-tail Coverage: {row['longtail_coverage']:.2f}%"
    )
    if has_pop_rank:
        txt += f"<br>Mean Popularity Rank: {row['mean_popularity_rank']:.1f}"
    return txt

hover_texts = [make_hover(row) for _, row in df_t.iterrows()]

# selected operating point
SELECTED_LAMBDA = 0.3
sel      = df_t[df_t['lambda'] == SELECTED_LAMBDA].iloc[0]
sel_idx  = df_t.index[df_t['lambda'] == SELECTED_LAMBDA][0]
mask_reg = df_t['lambda'] != SELECTED_LAMBDA
df_reg   = df_t[mask_reg].copy()
idxs_reg = df_reg.index.tolist()

# build figure
fig = go.Figure()

# 1. Dashed connector line (drawn first, behind markers)
fig.add_trace(go.Scatter(
    x=df_t['catalog_coverage'].tolist(),
    y=df_t['longtail_coverage'].tolist(),
    mode='lines',
    line=dict(color='rgba(120,120,120,0.55)', width=1.8, dash='dash'),
    hoverinfo='skip',
    showlegend=False,
))

# 2. Regular (non-selected) points — gradient-colored circles
fig.add_trace(go.Scatter(
    x=df_reg['catalog_coverage'].tolist(),
    y=df_reg['longtail_coverage'].tolist(),
    mode='markers',
    marker=dict(
        color=[pt_colors[i] for i in idxs_reg],
        size=16,
        line=dict(color='white', width=2),
        symbol='circle',
    ),
    hovertemplate='%{customdata}<extra></extra>',
    customdata=[hover_texts[i] for i in idxs_reg],
    showlegend=False,
))

# 3. Selected operating point — larger star marker
fig.add_trace(go.Scatter(
    x=[sel['catalog_coverage']],
    y=[sel['longtail_coverage']],
    mode='markers',
    marker=dict(
        color=pt_colors[sel_idx],
        size=26,
        symbol='star',
        line=dict(color='black', width=2),
    ),
    hovertemplate=hover_texts[sel_idx] + '<extra></extra>',
    showlegend=True,
    name=f'Selected operating point (\u03bb={SELECTED_LAMBDA})',
))

# 4. Annotation with arrow pointing to selected point
fig.add_annotation(
    x=sel['catalog_coverage'],
    y=sel['longtail_coverage'],
    ax=65, ay=-60,
    text=f'<b>Selected operating point</b><br>\u03bb = {SELECTED_LAMBDA}',
    showarrow=True,
    arrowhead=2,
    arrowsize=1.3,
    arrowwidth=2,
    arrowcolor='black',
    bgcolor='rgba(255,255,255,0.88)',
    bordercolor='black',
    borderwidth=1.5,
    borderpad=5,
    font=dict(size=12),
)

# 5. Lambda labels next to each non-selected point
for i, row in df_t[mask_reg].iterrows():
    fig.add_annotation(
        x=row['catalog_coverage'],
        y=row['longtail_coverage'],
        text=f"\u03bb={row['lambda']}",
        showarrow=False,
        xanchor='left',
        yanchor='middle',
        xshift=14,
        font=dict(size=10, color='rgba(50,50,50,0.9)'),
    )

# 6. Invisible scatter used solely to render the colorbar
fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(
        colorscale=[[0, lerp_color(0.0)], [0.5, lerp_color(0.5)], [1, lerp_color(1.0)]],
        color=[0],
        colorbar=dict(
            title=dict(text='\u03bb', font=dict(size=13)),
            thickness=14,
            len=0.55,
            y=0.5,
            tickvals=[0, 0.5, 1],
            ticktext=[f'{lam_min}', f'{(lam_min + lam_max) / 2:.2g}', f'{lam_max}'],
            outlinewidth=1,
        ),
        showscale=True,
        cmin=0, cmax=1,
    ),
    showlegend=False,
    hoverinfo='skip',
))

# layout
fig.update_layout(
    title=dict(
        text=(
            'Accuracy\u2013Fairness Tradeoff Curve<br>'
            '<sup>Each point = one penalty strength \u03bb  \u00b7  '
            'Blue = low \u03bb (accuracy-focused), Red = high \u03bb (fairness-focused)</sup>'
        ),
        x=0.5,
        font_size=15,
    ),
    xaxis=dict(
        title='Catalogue Coverage (%)',
        gridcolor='rgba(200,200,200,0.4)',
        showgrid=True,
        zeroline=False,
        title_font_size=13,
    ),
    yaxis=dict(
        title='Long-tail Coverage (%)',
        gridcolor='rgba(200,200,200,0.4)',
        showgrid=True,
        zeroline=False,
        title_font_size=13,
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.18,
        xanchor='center',
        x=0.42,
        font_size=12,
    ),
    hovermode='closest',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=560,
    width=820,
    margin=dict(t=105, b=110, l=70, r=110),
)

# save
out_path = OUTPUT_DIR + 'lambda_tradeoff.html'
fig.write_html(out_path, include_plotlyjs='cdn')
print(f'Saved  \u2192  {out_path}')

fig.show()

# Genre Representation: Long Tail vs. Recommendations by Model

Per-genre: % of long-tail films vs. % of recommendation slots across SVD, content-based, and hybrid. Saved to `outputs/genre_suppression.html`.

In [ ]:
import numpy as np
import pandas as pd
import joblib
import random
import plotly.graph_objects as go
from collections import defaultdict

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

OUTPUT_DIR  = '../outputs/'
DATA_DIR    = '../../ml-25m/'
RANDOM_SEED  = 42
TOP_N        = 10
N_EVAL_USERS = 500
ALPHA        = 0.6   # hybrid weight on content-based

# load models
print('Loading models...')
svd_model    = joblib.load(OUTPUT_DIR + 'svd_model.pkl')
cb_art       = joblib.load(OUTPUT_DIR + 'content_based_model.pkl')
tfidf_matrix  = cb_art['tfidf_matrix']
movie_ids_cb  = cb_art['movie_ids']
mid_to_idx_cb = cb_art['movie_id_to_idx']

# genre lookup from movies.csv
print('Loading movies.csv...')
movies_df    = pd.read_csv(DATA_DIR + 'movies.csv')
movie_genres = (movies_df.set_index('movieId')['genres']
                .apply(lambda s: [g for g in s.split('|') if g != '(no genres listed)'])
                .to_dict())
all_genres = sorted({g for gs in movie_genres.values() for g in gs})
print(f'  {len(movies_df):,} movies  |  {len(all_genres)} genres')

# long-tail definition: 80/20 rule on ratings volume
print('Computing tail from ratings.csv...')
ratings_df = pd.read_csv(DATA_DIR + 'ratings.csv', usecols=['movieId', 'rating'])
film_pop = (ratings_df.groupby('movieId').size()
            .reset_index(name='cnt')
            .sort_values('cnt', ascending=False)
            .reset_index(drop=True))
film_pop['cumpct'] = film_pop['cnt'].cumsum() / film_pop['cnt'].sum() * 100
cutoff   = (film_pop['cumpct'] >= 80).idxmax()
tail_ids = set(film_pop.loc[cutoff + 1:, 'movieId'])
print(f'  Head: {cutoff + 1:,} films  |  Tail: {len(tail_ids):,} films')

# genre distribution in the tail
tail_g_cnt = defaultdict(int)
n_tail_films = 0
for mid in tail_ids:
    gs = movie_genres.get(mid, [])
    if not gs:
        continue
    for g in gs:
        tail_g_cnt[g] += 1
    n_tail_films += 1
tail_pct = {g: tail_g_cnt[g] / n_tail_films * 100 for g in all_genres}

# SVD index structures
print('Building SVD lookup tables...')
ts = svd_model.trainset
inner_to_raw     = np.array([int(ts.to_raw_iid(i)) for i in range(ts.n_items)], dtype=np.int64)
raw_uid_to_inner = {int(ts.to_raw_uid(u)): u for u in range(ts.n_users)}

# sample 500 eval users from SVD trainset
random.seed(RANDOM_SEED)
eval_users = random.sample(list(raw_uid_to_inner.keys()), min(N_EVAL_USERS, ts.n_users))
print(f'Eval users: {len(eval_users):,}  '
      f'(SVD trainset: {ts.n_users:,} users, {ts.n_items:,} items, '
      f'{ts.n_ratings:,} ratings)')

# generate top-N recs and tally genre appearances
rec_g_cnt = {m: defaultdict(int) for m in ['svd', 'content_based', 'hybrid']}
rec_total  = {m: 0              for m in ['svd', 'content_based', 'hybrid']}

for uid in tqdm(eval_users, desc='Generating recs'):
    inner_uid = raw_uid_to_inner[uid]
    seen      = {int(ts.to_raw_iid(i)) for i, _ in ts.ur[inner_uid]}

    # SVD: batch score all items in one matrix-vector multiply
    svd_sc = (ts.global_mean
              + svd_model.bu[inner_uid]
              + svd_model.bi
              + svd_model.qi.dot(svd_model.pu[inner_uid]))

    recs_svd = sorted(
        ((int(inner_to_raw[i]), float(svd_sc[i]))
         for i in range(len(inner_to_raw)) if int(inner_to_raw[i]) not in seen),
        key=lambda x: x[1], reverse=True
    )[:TOP_N]

    # normalised SVD scores for hybrid fusion
    svd_norm = {int(inner_to_raw[i]): (float(svd_sc[i]) - 0.5) / 4.5
                for i in range(len(inner_to_raw))}

    # content-based: vectorised TF-IDF profile build
    cb_idx, cb_wts = [], []
    for i_inn, r in ts.ur[inner_uid]:
        mid = int(ts.to_raw_iid(i_inn))
        if mid in mid_to_idx_cb:
            cb_idx.append(mid_to_idx_cb[mid])
            cb_wts.append(r / 5.0)

    cb_scores = {}
    if cb_idx:
        w       = np.array(cb_wts, dtype=np.float32)
        profile = np.asarray(tfidf_matrix[cb_idx].T.dot(w)).ravel()
        norm    = np.linalg.norm(profile)
        if norm > 0:
            profile /= norm
            sims     = tfidf_matrix.dot(profile)
            cb_scores = {int(movie_ids_cb[i]): float(sims[i])
                         for i in range(len(movie_ids_cb))
                         if int(movie_ids_cb[i]) not in seen}

    recs_cb = sorted(cb_scores.items(), key=lambda x: x[1], reverse=True)[:TOP_N]

    # hybrid: union of SVD + CB candidates
    cands    = (set(svd_norm) | set(cb_scores)) - seen
    hyb_sc   = {m: (1 - ALPHA) * svd_norm.get(m, 0.) + ALPHA * cb_scores.get(m, 0.)
                for m in cands}
    recs_hyb = sorted(hyb_sc.items(), key=lambda x: x[1], reverse=True)[:TOP_N]

    # tally genre appearances for each model
    for mname, recs in [('svd', recs_svd), ('content_based', recs_cb), ('hybrid', recs_hyb)]:
        for mid, _ in recs:
            gs = movie_genres.get(mid, [])
            for g in gs:
                rec_g_cnt[mname][g] += 1
            rec_total[mname] += 1

print(f'\nRec totals: {rec_total}')

# compute rec_pct
rec_pct = {
    m: {g: rec_g_cnt[m][g] / rec_total[m] * 100 if rec_total[m] > 0 else 0.0
        for g in all_genres}
    for m in rec_g_cnt
}

# sort genres by tail_pct descending (most long-tail-represented genre at top)
genres_sorted = sorted(all_genres, key=lambda g: tail_pct.get(g, 0), reverse=True)

print('\nGenre breakdown (tail_pct | svd | content_based | hybrid):')
for g in genres_sorted:
    print(f'  {g:<15} tail={tail_pct[g]:5.1f}%  '
          f'svd={rec_pct["svd"][g]:5.1f}%  '
          f'cb={rec_pct["content_based"][g]:5.1f}%  '
          f'hyb={rec_pct["hybrid"][g]:5.1f}%')

In [ ]:
# horizontal grouped bar chart
BAR_COLOURS = {
    'tail'         : 'rgba(108, 117, 125, 0.80)',   # neutral grey
    'svd'          : 'rgba(239,  85,  59, 0.85)',   # red-orange
    'content_based': 'rgba(171,  99, 250, 0.85)',   # purple
    'hybrid'       : 'rgba(255, 161,  90, 0.85)',   # amber
}
BAR_LABELS = {
    'tail'         : 'Long Tail',
    'svd'          : 'SVD',
    'content_based': 'Content-Based',
    'hybrid'       : 'Hybrid (\u03b1=0.6)',
}

fig = go.Figure()

for key in ['tail', 'svd', 'content_based', 'hybrid']:
    if key == 'tail':
        vals = [tail_pct.get(g, 0.0) for g in genres_sorted]
    else:
        vals = [rec_pct[key].get(g, 0.0) for g in genres_sorted]

    hover = [
        f'<b>{g}</b> \u2014 {BAR_LABELS[key]}<br>'
        f'<b>{v:.1f}%</b> of {"tail films" if key == "tail" else "recommendation slots"}'
        f' belong to this genre'
        for g, v in zip(genres_sorted, vals)
    ]

    fig.add_trace(go.Bar(
        x=vals,
        y=genres_sorted,
        name=BAR_LABELS[key],
        orientation='h',
        marker=dict(color=BAR_COLOURS[key], line=dict(width=0)),
        hovertemplate='%{customdata}<extra></extra>',
        customdata=hover,
    ))

fig.update_layout(
    barmode='group',
    title=dict(
        text=(
            'Genre Representation: Long Tail vs Recommendations by Model<br>'
            '<sup>% of films / recommendation slots belonging to each genre \u00b7 '
            'genres sorted by long-tail prevalence \u00b7 '
            'multi-genre films counted once per genre</sup>'
        ),
        x=0.5,
        font_size=14,
    ),
    xaxis=dict(
        title='Percentage (%)',
        gridcolor='rgba(200,200,200,0.4)',
        showgrid=True,
        zeroline=False,
        title_font_size=12,
    ),
    yaxis=dict(
        title='Genre',
        autorange='reversed',
        tickfont=dict(size=12),
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.14,
        xanchor='center',
        x=0.5,
        font_size=12,
        itemsizing='constant',
    ),
    hovermode='closest',
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=55 * len(genres_sorted) + 200,
    width=900,
    margin=dict(t=110, b=120, l=120, r=40),
)

out_path = OUTPUT_DIR + 'genre_suppression.html'
fig.write_html(out_path, include_plotlyjs='cdn')
print(f'Saved  \u2192  {out_path}')

fig.show()

# Model Comparison Dashboard (2×2)

NDCG@10 bars, coverage scatter, hybrid α sweep, and radar chart. Saved to `outputs/model_dashboard.html`.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUTPUT_DIR = '../outputs/'

# load data
summary     = pd.read_csv(OUTPUT_DIR + 'hybrid_summary.csv')
fairness    = pd.read_csv(OUTPUT_DIR + 'fairness_comparison.csv')
alpha_sweep = pd.read_csv(OUTPUT_DIR + 'hybrid_alpha_sweep.csv')

# model metadata
MODEL_ORDER = ['popularity_baseline', 'svd', 'penalized_svd',
               'content_based', 'hybrid_weighted (α=0.6)', 'hybrid_rrf']
MODEL_LABEL = {
    'popularity_baseline'    : 'Pop. Baseline',
    'svd'                    : 'SVD',
    'penalized_svd'          : 'Penalised SVD',
    'content_based'          : 'Content-Based',
    'hybrid_weighted (α=0.6)': 'Hybrid (α=0.6)',
    'hybrid_rrf'             : 'Hybrid RRF',
}
MODEL_COLOUR = {
    'popularity_baseline'    : '#636EFA',
    'svd'                    : '#EF553B',
    'penalized_svd'          : '#00CC96',
    'content_based'          : '#AB63FA',
    'hybrid_weighted (α=0.6)': '#FFA15A',
    'hybrid_rrf'             : '#19D3F3',
}

summary = summary.set_index('model').reindex(MODEL_ORDER).reset_index()
summary['label']  = summary['model'].map(MODEL_LABEL)
summary['colour'] = summary['model'].map(MODEL_COLOUR)

# mean Popularity Rank (only SVD & Penalised SVD have this metric)
fair_idx = fairness.set_index('Metric')
pop_rank_map = {
    'svd'          : float(fair_idx.loc['Mean Popularity Rank', 'SVD Baseline']),
    'penalized_svd': float(fair_idx.loc['Mean Popularity Rank', 'Penalized (\u03bb=0.3)']),
}
summary['mean_pop_rank'] = summary['model'].map(pop_rank_map)
summary['inv_pop_rank']  = 1.0 - summary['mean_pop_rank']  # higher = less biased

# normalise all four radar dimensions to [0, 1]
def minmax(series):
    lo, hi = series.min(), series.max()
    return (series - lo) / (hi - lo) if hi > lo else series * 0 + 1.0

summary['ndcg_norm']   = minmax(summary['ndcg10'])
summary['catcov_norm'] = minmax(summary['cat_cov'])
summary['ltcov_norm']  = minmax(summary['lt_cov'])

# inv pop rank: normalise over available values only; unknown models → 0 (N/A)
known_pr     = summary['inv_pop_rank'].dropna()
pr_lo, pr_hi = known_pr.min(), known_pr.max()
def norm_pr(v):
    if pd.isna(v): return 0.0
    return float((v - pr_lo) / (pr_hi - pr_lo)) if pr_hi > pr_lo else 1.0
summary['pr_norm'] = summary['inv_pop_rank'].apply(norm_pr)

print('Summary with normalised dimensions:')
print(summary[['label', 'ndcg10', 'cat_cov', 'lt_cov', 'mean_pop_rank',
               'ndcg_norm', 'catcov_norm', 'ltcov_norm', 'pr_norm']].to_string(index=False))

# build 2×2 dashboard
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        '<b>NDCG@10 by Model</b>',
        '<b>Catalogue vs Long-Tail Coverage</b>',
        '<b>Hybrid: NDCG@10 & Coverage vs \u03b1</b>',
        '<b>Model Radar \u2014 Normalised Scores (0\u21921)</b>',
    ],
    specs=[
        [{'type': 'xy'},    {'type': 'xy'}   ],
        [{'type': 'xy'},    {'type': 'polar'}],
    ],
    vertical_spacing=0.22,
    horizontal_spacing=0.13,
)

# panel 1 (1,1): NDCG@10 bars + asymmetric error bars
for _, row in summary.iterrows():
    err_hi = max(0.0, float(row['ci_hi']) - float(row['ndcg10']))
    err_lo = max(0.0, float(row['ndcg10']) - float(row['ci_lo']))
    fig.add_trace(go.Bar(
        x=[row['label']], y=[row['ndcg10']],
        error_y=dict(type='data', symmetric=False,
                     array=[err_hi], arrayminus=[err_lo],
                     color='rgba(50,50,50,0.6)', thickness=2, width=6),
        marker=dict(color=row['colour'], line=dict(width=0)),
        name=row['label'],
        legendgroup=row['model'],
        showlegend=True,
        hovertemplate=(
            f"<b>{row['label']}</b><br>"
            f"NDCG@10: {row['ndcg10']:.4f}<br>"
            f"95% CI: [{row['ci_lo']:.4f}, {row['ci_hi']:.4f}]"
            "<extra></extra>"
        ),
    ), row=1, col=1)

# panel 2 (1,2): Coverage scatter — labeled point per model
TEXT_POS = ['top center', 'bottom center', 'top right',
            'top left', 'bottom right', 'bottom left']
for i, (_, row) in enumerate(summary.iterrows()):
    fig.add_trace(go.Scatter(
        x=[row['cat_cov']], y=[row['lt_cov']],
        mode='markers+text',
        text=[row['label']],
        textposition=TEXT_POS[i % len(TEXT_POS)],
        textfont=dict(size=10),
        marker=dict(color=row['colour'], size=14,
                    line=dict(color='white', width=1.5)),
        name=row['label'],
        legendgroup=row['model'],
        showlegend=False,
        hovertemplate=(
            f"<b>{row['label']}</b><br>"
            f"Catalogue Coverage: {row['cat_cov']:.2f}%<br>"
            f"Long-Tail Coverage: {row['lt_cov']:.2f}%"
            "<extra></extra>"
        ),
    ), row=1, col=2)

# panel 3 (2,1): Alpha sweep — NDCG@10 with CI + coverage lines
x_fwd, x_rev = alpha_sweep['alpha'].tolist(), alpha_sweep['alpha'].tolist()[::-1]

# CI band
fig.add_trace(go.Scatter(
    x=x_fwd + x_rev,
    y=alpha_sweep['ci_hi'].tolist() + alpha_sweep['ci_lo'].tolist()[::-1],
    fill='toself', fillcolor='rgba(99,110,250,0.15)',
    line=dict(width=0), hoverinfo='skip', showlegend=False,
), row=2, col=1)

# NDCG@10 line
fig.add_trace(go.Scatter(
    x=alpha_sweep['alpha'], y=alpha_sweep['ndcg10'],
    mode='lines+markers', name='NDCG@10 (\u03b1 sweep)',
    line=dict(color='#636EFA', width=2.5),
    marker=dict(size=7, color='#636EFA', line=dict(color='white', width=1)),
    legendgroup='alpha_ndcg', showlegend=True,
    hovertemplate=(
        '<b>\u03b1 = %{x:.1f}</b><br>NDCG@10: %{y:.4f}'
        '<extra>NDCG@10</extra>'
    ),
), row=2, col=1)

# catalogue coverage line
fig.add_trace(go.Scatter(
    x=alpha_sweep['alpha'], y=alpha_sweep['cat_cov'],
    mode='lines+markers', name='Cat. Coverage % (\u03b1 sweep)',
    line=dict(color='#AB63FA', width=2, dash='dot'),
    marker=dict(size=6, color='#AB63FA'),
    legendgroup='alpha_cat', showlegend=True,
    hovertemplate=(
        '<b>\u03b1 = %{x:.1f}</b><br>Catalogue Coverage: %{y:.2f}%'
        '<extra>Cat. Coverage</extra>'
    ),
), row=2, col=1)

# long-tail coverage line
fig.add_trace(go.Scatter(
    x=alpha_sweep['alpha'], y=alpha_sweep['lt_cov'],
    mode='lines+markers', name='LT Coverage % (\u03b1 sweep)',
    line=dict(color='#FFA15A', width=2, dash='dash'),
    marker=dict(size=6, color='#FFA15A'),
    legendgroup='alpha_lt', showlegend=True,
    hovertemplate=(
        '<b>\u03b1 = %{x:.1f}</b><br>Long-Tail Coverage: %{y:.2f}%'
        '<extra>LT Coverage</extra>'
    ),
), row=2, col=1)

# mark selected α=0.6
fig.add_vline(x=0.6, line=dict(color='black', dash='dash', width=1.2),
              annotation_text='\u03b1=0.6<br>(selected)', annotation_font_size=9,
              annotation_position='top right', row=2, col=1)

# panel 4 (2,2): Radar — 4 normalised dimensions, all 6 models
RADAR_DIMS = ['NDCG@10', 'Cat. Coverage', 'LT Coverage', 'Inv. Pop. Rank']

for _, row in summary.iterrows():
    r_vals = [row['ndcg_norm'], row['catcov_norm'],
               row['ltcov_norm'], row['pr_norm']]
    r_closed    = r_vals + [r_vals[0]]
    dims_closed = RADAR_DIMS + [RADAR_DIMS[0]]

    has_pr  = not pd.isna(row['mean_pop_rank'])
    pr_line = (f"Inv. Pop. Rank: {row['inv_pop_rank']:.3f} (norm {row['pr_norm']:.2f})"
               if has_pr else "Inv. Pop. Rank: N/A (not computed for this model)")
    hover = (
        f"<b>{row['label']}</b><br>"
        f"NDCG@10: {row['ndcg10']:.4f} \u2192 norm {row['ndcg_norm']:.2f}<br>"
        f"Cat. Coverage: {row['cat_cov']:.2f}% \u2192 norm {row['catcov_norm']:.2f}<br>"
        f"LT Coverage: {row['lt_cov']:.2f}% \u2192 norm {row['ltcov_norm']:.2f}<br>"
        f"{pr_line}"
    )
    c    = row['colour']
    rgba = (f"rgba({int(c[1:3],16)},"
            f"{int(c[3:5],16)},"
            f"{int(c[5:7],16)},0.12)")

    fig.add_trace(go.Scatterpolar(
        r=r_closed, theta=dims_closed,
        fill='toself', fillcolor=rgba,
        line=dict(color=row['colour'], width=2),
        name=row['label'],
        legendgroup=row['model'],
        showlegend=False,
        hovertemplate=hover + '<extra></extra>',
    ), row=2, col=2)

# global layout
fig.update_layout(
    title=dict(
        text=(
            'Model Comparison Dashboard<br>'
            '<sup>Top: NDCG@10 bars | Coverage scatter  \u00b7  '
            'Bottom: Hybrid \u03b1 sweep | Normalised radar '
            '(Inv. Pop. Rank only for SVD & Penalised SVD)</sup>'
        ),
        x=0.5, font_size=15,
    ),
    barmode='group',
    plot_bgcolor='white', paper_bgcolor='white',
    height=940, width=1220,
    margin=dict(t=115, b=80, l=60, r=60),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.09,
        xanchor='center', x=0.5,
        font_size=11, itemsizing='constant', tracegroupgap=4,
    ),
    hovermode='closest',
)

# axis labels & grid
grid_style = dict(showgrid=True, gridcolor='rgba(200,200,200,0.4)', zeroline=False)
fig.update_xaxes(title_text='Model', **grid_style, row=1, col=1)
fig.update_yaxes(title_text='NDCG@10', showgrid=True,
                 gridcolor='rgba(200,200,200,0.4)',
                 zeroline=True, zerolinecolor='rgba(150,150,150,0.4)', row=1, col=1)
fig.update_xaxes(title_text='Catalogue Coverage (%)', **grid_style, row=1, col=2)
fig.update_yaxes(title_text='Long-Tail Coverage (%)', showgrid=True,
                 gridcolor='rgba(200,200,200,0.4)',
                 zeroline=True, zerolinecolor='rgba(150,150,150,0.4)', row=1, col=2)
fig.update_xaxes(title_text='Content-Based Weight \u03b1',
                 dtick=0.1, **grid_style, row=2, col=1)
fig.update_yaxes(title_text='Value', showgrid=True,
                 gridcolor='rgba(200,200,200,0.4)', zeroline=False, row=2, col=1)

fig.update_polars(
    radialaxis=dict(range=[0, 1.05], showticklabels=True,
                    tickfont_size=8, gridcolor='rgba(180,180,180,0.5)'),
    angularaxis=dict(tickfont_size=10),
    bgcolor='white',
)

# save
out_path = OUTPUT_DIR + 'model_dashboard.html'
fig.write_html(out_path, include_plotlyjs='cdn')
print(f'Saved  \u2192  {out_path}')

fig.show()